# Study 873 — Sentiment Beta — the teardown

The per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the post-peak conditional, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 3831, 'fingerprint': '357fd262912f', 'gauge_bps': 4.74, 'gauge_annvol': 19.4, 'gauge_ac': -0.007, 'spread_bps': -6.2, 't_nw': -2.86, 't_1s': -2.77, 'lo_bps': 4.82, 'hi_bps': 11.01, 'welch_t': -2.05, 'gross_sharpe': -0.71, 'cond_high_bps': -9.21, 'cond_high_t': -2.48, 'cond_high_n': 1150, 'cond_rest_bps': -4.91, 'cond_rest_t': -1.86, 'cond_rest_n': 2681, 'placebo_obs': -6.2, 'placebo_mean': 0.004, 'placebo_sd': 1.317, 'placebo_p': 1.0, 'placebo_sigma_left': 4.71, 'placebo_draws': 1000, 'era_early_bps': -4.45, 'era_early_t': -1.73, 'era_early_n': 1697, 'era_late_bps': -7.59, 'era_late_t': -2.29, 'era_late_n': 2134, 'timer_1_gross': -6.2, 'timer_1_cost': 2.14, 'timer_1_net': -8.34, 'timer_1_t': -3.73, 'timer_5_gross': -6.2, 'timer_5_cost': 10.14, 'timer_5_net': -16.34, 'timer_5_t': -7.3, 'null_mean_t': -0.05, 'null_sd_t': 0.94, 'null_fire': 0, 'planted_t': 5.86, 'planted_welch': 8.13}

## The sentiment gauge — a tradable high-minus-low-vol spread

Daily return of the top-30% by trailing-63d vol (speculative) minus the bottom-30% (safe). Real data, built from the panel; low autocorrelation.

In [2]:
print(f"gauge         : {R['gauge_bps']:+.2f} bps/day  ann-vol {R['gauge_annvol']:.1f}%  "
      f"lag-1 autocorr {R['gauge_ac']:+.3f}")

gauge         : +4.74 bps/day  ann-vol 19.4%  lag-1 autocorr -0.007


## The headline — long-low-beta / short-high-beta spread

Daily equal-weight bottom-30% minus top-30% sentiment-beta spread.

In [3]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : low-beta {R['lo_bps']:+.2f} vs high-beta {R['hi_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : -6.20 bps/day  NW(10) t = -2.86  one-sample t = -2.77
books         : low-beta +4.82 vs high-beta +11.01 bps (Welch t = -2.05)
gross Sharpe  : -0.71 (before cost)


## Conditional — is it stronger *after sentiment peaks*?

Split the spread by the trailing gauge level (top-30% = a high-sentiment regime).

In [4]:
print(f"high-sentiment (after peaks, n={R['cond_high_n']}): {R['cond_high_bps']:+.2f} bps  NW t = {R['cond_high_t']:+.2f}")
print(f"the rest                 (n={R['cond_rest_n']}): {R['cond_rest_bps']:+.2f} bps  NW t = {R['cond_rest_t']:+.2f}")
print('-> the conditioning works, but with the REVERSED sign (high-beta out-earned MORE after peaks)')

high-sentiment (after peaks, n=1150): -9.21 bps  NW t = -2.48
the rest                 (n=2681): -4.91 bps  NW t = -1.86
-> the conditioning works, but with the REVERSED sign (high-beta out-earned MORE after peaks)


## Placebo — column-permute the forward returns (1,000 permutations)

In [5]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> ~{R['placebo_sigma_left']:.1f} sigma into the LEFT tail (right-tail p = {R['placebo_p']:.5f})")

observed -6.20 bps vs placebo mean +0.004 (sd 1.317) -> ~4.7 sigma into the LEFT tail (right-tail p = 1.00000)


## Robustness — two eras (split 2018-01-01)

In [6]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1697): -4.45 bps  NW t = -1.73
2018-2026 (n=2134): -7.59 bps  NW t = -2.29


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [7]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross -6.20 -> net -8.34 bps/day (cost 2.14/day, t=-3.73)
5 bps one-way: gross -6.20 -> net -16.34 bps/day (cost 10.14/day, t=-7.30)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted relation.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from sentiment_beta import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=873+s, n_assets=40, n_days=1400))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0025, seed=873, n_assets=40, n_days=1600))
print(f"planted (edge=0.0025): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean -0.33 (sd 1.02), |t|>=2 in 0/8


planted (edge=0.0025): NW t = +5.86, Welch t = +8.13


## Verdict

- **Signal — None.** The claimed Baker-Wurgler sentiment-beta premium does **not** replicate on 50 liquid US mega-caps: the long-low-beta / short-high-beta spread is **-6.20 bps/day** (NW *t* = **-2.86**) — significant but *opposite in sign* (the permutation null centres at 0, sd 1.32 bps; observed ~4.7σ into the left tail), and the reversal is present in both eras (*t* = -1.73 / -2.29). The 20-seed synthetic control recovers a *planted* relation cleanly (*t* = +5.86, fires on 0/20 nulls), so the sign-reversal is real, not machinery. Survivorship biases the magnitude.
- **Tradability — Mirage.** Even the sign-flipped book dies: at 1 bp one-way the friction (2.14 bps/day) already erodes the 6.20 bps gross edge, net **-8.34 bps/day** (*t* = -3.73); at 5 bps **-16.34 bps/day**.